# HAR-RV: Modelo Heterogeneo de Volatilidade Realizada

Neste notebook, exploraremos o modelo **HAR-RV** (Heterogeneous Autoregressive
Realized Volatility), proposto por **Corsi (2009)**.

O HAR-RV e uma alternativa simples e poderosa aos modelos GARCH para prever
volatilidade. Ele usa **volatilidade realizada** (calculada a partir de dados
intradiarios) em tres horizontes temporais como regressores.

**Conteudo:**
1. Volatilidade realizada
2. Modelo HAR de Corsi (2009)
3. Heterogeneidade de agentes
4. HAR-RV vs GARCH
5. Extensoes: HAR-RV-J

**Referencias:**
- Corsi, F. (2009). *A simple approximate long-memory model of realized volatility*. Journal of Financial Econometrics, 7(2), 174-196.
- Andersen, T.G., Bollerslev, T., Diebold, F.X. & Labys, P. (2003). *Modeling and forecasting realized volatility*. Econometrica, 71(2), 579-625.
- Baillie, R.T., Bollerslev, T. & Mikkelsen, H.O. (1996). *Fractionally integrated generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 74(1), 3-30.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from utils.plot_helpers import plot_har_components, plot_realized_vs_conditional

from archbox.models import GARCH, HARRV

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Volatilidade realizada

A **volatilidade realizada** (RV) e uma medida ex-post da volatilidade, calculada
a partir de retornos intradiarios de alta frequencia:

$$RV_t^{(d)} = \sum_{i=1}^{M} r_{t,i}^2$$

onde $r_{t,i}$ sao os $M$ retornos intradiarios do dia $t$ (ex: retornos de 5 minutos).

Sob certas condicoes, a RV e um estimador **consistente** da variancia integrada:

$$RV_t^{(d)} \xrightarrow{p} \int_0^1 \sigma_t^2(s) ds \quad \text{quando } M \to \infty$$

O modelo HAR-RV utiliza tres componentes temporais:
- **Diaria**: $RV_t^{(d)}$ — volatilidade do ultimo dia
- **Semanal**: $RV_t^{(w)} = \frac{1}{5} \sum_{i=0}^{4} RV_{t-i}^{(d)}$ — media dos ultimos 5 dias
- **Mensal**: $RV_t^{(m)} = \frac{1}{22} \sum_{i=0}^{21} RV_{t-i}^{(d)}$ — media dos ultimos 22 dias

In [ ]:
# Carregar dados de volatilidade realizada
rv_data = pd.read_csv('../data/realized_volatility.csv', parse_dates=['date'], index_col='date')
rv_daily = rv_data['rv_daily']
rv_weekly = rv_data['rv_weekly']
rv_monthly = rv_data['rv_monthly']

# Visualizar as 3 componentes usando plot_har_components
fig = plot_har_components(rv_data.index, rv_daily.values, rv_weekly.values, rv_monthly.values)
plt.show()

# Estatisticas descritivas de cada componente
desc = pd.DataFrame({
    'RV Daily': rv_daily.describe(),
    'RV Weekly': rv_weekly.describe(),
    'RV Monthly': rv_monthly.describe(),
})
print('=== Estatisticas Descritivas ===')
print(desc.to_string(float_format='%.6f'))

print('\nObserve: RV mensal e mais suave (std menor) devido ao efeito da media.')
print(f'Correlacao Daily-Weekly:  {np.corrcoef(rv_daily, rv_weekly)[0,1]:.4f}')
print(f'Correlacao Daily-Monthly: {np.corrcoef(rv_daily, rv_monthly)[0,1]:.4f}')
print(f'Correlacao Weekly-Monthly: {np.corrcoef(rv_weekly, rv_monthly)[0,1]:.4f}')

## 2. Modelo HAR de Corsi (2009)

O modelo HAR-RV e uma regressao linear simples:

$$RV_{t+1}^{(d)} = \beta_0 + \beta_d \cdot RV_t^{(d)} + \beta_w \cdot RV_t^{(w)} + \beta_m \cdot RV_t^{(m)} + \varepsilon_{t+1}$$

**Propriedades:**
- Estimacao por **OLS** (minimos quadrados ordinarios) — rapido e simples
- Apesar de ser um modelo AR(22) restrito, captura **memoria longa** na volatilidade
- A inclusao de componentes em multiplos horizontes gera um decaimento lento
  da funcao de autocorrelacao, mimetizando processos de memoria longa

Os coeficientes $\beta_d$, $\beta_w$, $\beta_m$ capturam a importancia relativa
de cada horizonte temporal para prever a volatilidade futura.

In [ ]:
# Estimar HAR-RV por OLS usando archbox
model_har = HARRV(rv_daily.values)
results_har = model_har.fit()

# Exibir resultados
print(results_har.summary())

# R-squared e coeficientes
print(f'\nR-squared: {results_har.r_squared:.6f}')
print(f'Adj. R-squared: {results_har.adj_r_squared:.6f}')
print(f'N. observacoes: {results_har.nobs}')

# Verificar significancia dos coeficientes
print('\n--- Significancia dos Coeficientes ---')
for name, tval in zip(results_har.param_names, results_har.t_values, strict=False):
    sig = '***' if abs(tval) > 2.576 else '**' if abs(tval) > 1.96 else '*' if abs(tval) > 1.645 else ''
    print(f'{name:>10}: t = {tval:>8.4f} {sig}')
print('\n(*** p<0.01, ** p<0.05, * p<0.10)')

## 3. Heterogeneidade de agentes

A interpretacao economica do HAR-RV baseia-se na **Hipotese de Mercados Heterogeneos**
(Muller et al., 1997):

| Componente | Horizonte | Tipo de agente | Exemplos |
|:---:|:---:|:---:|:---:|
| $\beta_d$ (diario) | 1 dia | Traders de alta frequencia | Day traders, market makers |
| $\beta_w$ (semanal) | 5 dias | Traders de media frequencia | Swing traders, fundos quantitativos |
| $\beta_m$ (mensal) | 22 dias | Investidores de longo prazo | Fundos de pensao, gestoras macro |

A volatilidade observada e resultado da **superposicao** das acoes de agentes
com diferentes horizontes de investimento. Cada grupo reage a informacao
em velocidades diferentes:
- Traders de alta frequencia reagem a choques recentes ($\beta_d$ alto)
- Investidores de longo prazo suavizam informacao ($\beta_m$ significativo)

In [ ]:
# Analisar coeficientes beta_d, beta_w, beta_m
beta_0 = results_har.params[0]
beta_d = results_har.params[1]
beta_w = results_har.params[2]
beta_m = results_har.params[3]

print('=== Coeficientes HAR-RV ===')
print(f'beta_0 (intercepto): {beta_0:.6f}')
print(f'beta_d (diario):     {beta_d:.6f}')
print(f'beta_w (semanal):    {beta_w:.6f}')
print(f'beta_m (mensal):     {beta_m:.6f}')
print(f'Soma (beta_d + beta_w + beta_m): {beta_d + beta_w + beta_m:.6f}')

# Grafico de barras dos coeficientes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Coeficientes
coef_names = ['beta_d\n(diario)', 'beta_w\n(semanal)', 'beta_m\n(mensal)']
coef_vals = [beta_d, beta_w, beta_m]
coef_ses = [results_har.std_errors[1], results_har.std_errors[2], results_har.std_errors[3]]
colors = ['steelblue', 'darkorange', 'darkgreen']

bars = axes[0].bar(coef_names, coef_vals, color=colors, alpha=0.8,
                    yerr=[1.96 * se for se in coef_ses], capsize=5)
axes[0].set_ylabel('Coeficiente')
axes[0].set_title('Coeficientes HAR-RV (com IC 95%)')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, coef_vals, strict=False):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# Subplot 2: Contribuicao relativa
# Contribuicao = coeficiente * desvio padrao do regressor
# Precisamos recalcular os regressores para obter os std
rv_vals = rv_daily.values
n = len(rv_vals)
rv_d_reg = rv_vals[21:-1]  # lag-1 daily
rv_w_reg = np.array([np.mean(rv_vals[i-5:i]) for i in range(22, n)])[:-1]
rv_m_reg = np.array([np.mean(rv_vals[i-22:i]) for i in range(22, n)])[:-1]

contrib_d = abs(beta_d) * np.std(rv_d_reg)
contrib_w = abs(beta_w) * np.std(rv_w_reg)
contrib_m = abs(beta_m) * np.std(rv_m_reg)
total_contrib = contrib_d + contrib_w + contrib_m

contrib_pct = [contrib_d/total_contrib*100, contrib_w/total_contrib*100, contrib_m/total_contrib*100]
axes[1].bar(coef_names, contrib_pct, color=colors, alpha=0.8)
axes[1].set_ylabel('Contribuicao Relativa (%)')
axes[1].set_title('Contribuicao Relativa de Cada Componente')
axes[1].grid(True, alpha=0.3, axis='y')
for i, (name, pct) in enumerate(zip(coef_names, contrib_pct, strict=False)):
    axes[1].text(i, pct + 0.5, f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print('\nContribuicao relativa (|beta| * std(regressor)):')
print(f'  Diario:  {contrib_pct[0]:.1f}%')
print(f'  Semanal: {contrib_pct[1]:.1f}%')
print(f'  Mensal:  {contrib_pct[2]:.1f}%')

## 4. HAR-RV vs GARCH

Vamos comparar o poder preditivo do HAR-RV com o GARCH(1,1):

| Aspecto | GARCH(1,1) | HAR-RV |
|:---:|:---:|:---:|
| Dados | Retornos diarios | Volatilidade realizada (intradiarios) |
| Estimacao | MLE (nao-linear) | OLS (linear) |
| Memoria | Curta (exponencial) | Aproxima longa (hiperbolica) |
| Volatilidade | Latente ($\sigma_t$) | Observavel ($RV_t$) |
| Vantagem | Nao requer dados intradiarios | Mais preciso, mais simples |

Para uma comparacao justa, usaremos a **raiz do erro quadratico medio (RMSE)**
das previsoes out-of-sample.

In [ ]:
# Carregar retornos para estimar GARCH
sp500 = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = sp500['returns']

# Estimar GARCH(1,1)
model_garch = GARCH(returns.values, p=1, q=1)
res_garch = model_garch.fit()

print('=== GARCH(1,1) ===')
print(res_garch.summary())

# Variancia condicional do GARCH
garch_var = res_garch.conditional_volatility ** 2

# Valores ajustados do HAR-RV
har_fitted = results_har.fitted_values

# Alinhar as series: HAR-RV comeca no lag 22
# rv_daily tem 2500 obs, har_fitted tem 2500-22 = 2478 obs
# A variavel dependente do HAR-RV e rv_daily[22:]
rv_actual = rv_daily.values[22:]
n_har = len(rv_actual)

# Para comparacao justa, usamos a variancia condicional do GARCH
# no mesmo periodo (ultimas n_har observacoes)
garch_var_aligned = garch_var[-n_har:]

# Calcular RMSE
rmse_har = np.sqrt(np.mean((rv_actual - har_fitted) ** 2))
rmse_garch = np.sqrt(np.mean((rv_actual - garch_var_aligned) ** 2))

# MAE
mae_har = np.mean(np.abs(rv_actual - har_fitted))
mae_garch = np.mean(np.abs(rv_actual - garch_var_aligned))

# Tabela comparativa
perf = pd.DataFrame({
    'Modelo': ['GARCH(1,1)', 'HAR-RV'],
    'RMSE': [rmse_garch, rmse_har],
    'MAE': [mae_garch, mae_har],
})
print('\n=== Comparacao de Previsao: GARCH vs HAR-RV ===')
print(perf.to_string(index=False, float_format='%.6f'))

# Visualizar com plot_realized_vs_conditional
dates_aligned = rv_data.index[22:]
fig = plot_realized_vs_conditional(
    dates=dates_aligned,
    rv_daily=rv_actual,
    conditional_vol=np.sqrt(np.maximum(har_fitted, 1e-12)),
    title='HAR-RV: Realizado vs Ajustado'
)
plt.show()

print(f'\nHAR-RV R-squared: {results_har.r_squared:.4f}')
if rmse_har < rmse_garch:
    print(f'HAR-RV tem RMSE menor que GARCH ({rmse_har:.6f} < {rmse_garch:.6f})')
else:
    print(f'GARCH tem RMSE menor que HAR-RV ({rmse_garch:.6f} < {rmse_har:.6f})')

## 5. Extensoes: HAR-RV-J

Uma extensao importante do HAR-RV e a inclusao de um componente de **jumps** (saltos):

$$RV_{t+1}^{(d)} = \beta_0 + \beta_d \cdot RV_t^{(d)} + \beta_w \cdot RV_t^{(w)} + \beta_m \cdot RV_t^{(m)} + \beta_j \cdot J_t + \varepsilon_{t+1}$$

O componente de jumps pode ser estimado como:

$$J_t = \max(RV_t^{(d)} - BV_t, 0)$$

onde $BV_t$ e a **bipower variation** (variacao bipotencia), que e robusta a jumps:

$$BV_t = \frac{\pi}{2} \sum_{i=2}^{M} |r_{t,i}| |r_{t,i-1}|$$

A ideia e que jumps tem um efeito **assimetrico** na volatilidade futura:
dias com jumps grandes podem sinalizar mudancas de regime.

**Nota**: Como nao temos a bipower variation nos dados, vamos aproximar
o componente de jumps usando a diferenca entre a RV diaria e a RV semanal
como proxy para movimentos anomalos.

In [ ]:
# Estimar HAR-RV-J com componente de jumps

# Proxy de jumps: max(RV_daily - RV_weekly, 0)
jumps = np.maximum(rv_daily.values - rv_weekly.values, 0)

# Construir regressao manualmente
rv_vals = rv_daily.values
n = len(rv_vals)
max_lag = 22
n_obs = n - max_lag

# Variavel dependente: RV_{t+1} (a partir do lag 22)
y = rv_vals[max_lag:]

# Regressores (lagged)
rv_d_lag = rv_vals[max_lag - 1:-1]  # RV_t^(d)
rv_w_lag = np.array([np.mean(rv_vals[i - 5:i]) for i in range(max_lag, n)])[:-1]
# Ajustar comprimento para alinhar com y
rv_d_lag = rv_d_lag[:len(y)]
rv_w_lag_arr = np.array([np.mean(rv_vals[max(0, i - 5):i]) for i in range(max_lag, n)])
rv_m_lag_arr = np.array([np.mean(rv_vals[max(0, i - 22):i]) for i in range(max_lag, n)])
jumps_lag_arr = jumps[max_lag - 1:-1] if len(jumps) > max_lag else jumps[:len(y)]

# Garantir alinhamento: usar [:-1] para lag e [1:] para y
y_har = rv_vals[max_lag + 1:]
rv_d_reg = rv_vals[max_lag:-1]
rv_w_reg = np.array([np.mean(rv_vals[i-5:i]) for i in range(max_lag + 1, n)])
rv_m_reg = np.array([np.mean(rv_vals[i-22:i]) for i in range(max_lag + 1, n)])
jumps_reg = jumps[max_lag:-1]

# Garantir tamanhos consistentes
min_len = min(len(y_har), len(rv_d_reg), len(rv_w_reg), len(rv_m_reg), len(jumps_reg))
y_har = y_har[:min_len]
rv_d_reg = rv_d_reg[:min_len]
rv_w_reg = rv_w_reg[:min_len]
rv_m_reg = rv_m_reg[:min_len]
jumps_reg = jumps_reg[:min_len]

# HAR-RV (sem jumps) por OLS manual
X_har = np.column_stack([np.ones(min_len), rv_d_reg, rv_w_reg, rv_m_reg])
beta_har = np.linalg.lstsq(X_har, y_har, rcond=None)[0]
fitted_har = X_har @ beta_har
resid_har = y_har - fitted_har
ss_res_har = np.sum(resid_har ** 2)
ss_tot = np.sum((y_har - np.mean(y_har)) ** 2)
r2_har = 1 - ss_res_har / ss_tot

# HAR-RV-J (com jumps) por OLS
X_harj = np.column_stack([np.ones(min_len), rv_d_reg, rv_w_reg, rv_m_reg, jumps_reg])
beta_harj = np.linalg.lstsq(X_harj, y_har, rcond=None)[0]
fitted_harj = X_harj @ beta_harj
resid_harj = y_har - fitted_harj
ss_res_harj = np.sum(resid_harj ** 2)
r2_harj = 1 - ss_res_harj / ss_tot

# Erros padrao e t-values para HAR-RV-J
k_j = X_harj.shape[1]
s2_j = ss_res_harj / (min_len - k_j)
var_beta_j = s2_j * np.linalg.inv(X_harj.T @ X_harj)
se_j = np.sqrt(np.diag(var_beta_j))
t_j = beta_harj / se_j
p_j = 2 * (1 - stats.t.cdf(np.abs(t_j), df=min_len - k_j))

# Tabela de resultados HAR-RV-J
harj_names = ['beta_0', 'beta_d', 'beta_w', 'beta_m', 'beta_j']
print('=== HAR-RV-J Regression Results ===')
print(f'{"Parameter":<15} {"Estimate":>12} {"Std.Err":>12} {"t-value":>12} {"p-value":>12}')
print('-' * 65)
for name, coef, se, tv, pv in zip(harj_names, beta_harj, se_j, t_j, p_j, strict=False):
    sig = '***' if pv < 0.01 else '**' if pv < 0.05 else '*' if pv < 0.10 else ''
    print(f'{name:<15} {coef:>12.6f} {se:>12.6f} {tv:>12.4f} {pv:>12.6f} {sig}')

# Comparacao R^2
print('\n=== Comparacao R-squared ===')
print(f'HAR-RV:   R^2 = {r2_har:.6f}')
print(f'HAR-RV-J: R^2 = {r2_harj:.6f}')
print(f'Melhoria:       {(r2_harj - r2_har) * 100:.4f} pontos percentuais')

# RMSE comparacao
rmse_har_manual = np.sqrt(np.mean(resid_har ** 2))
rmse_harj = np.sqrt(np.mean(resid_harj ** 2))
print(f'\nRMSE HAR-RV:   {rmse_har_manual:.6f}')
print(f'RMSE HAR-RV-J: {rmse_harj:.6f}')

beta_j_val = beta_harj[4]
if p_j[4] < 0.05:
    print(f'\nCoeficiente de jumps (beta_j = {beta_j_val:.6f}) e significativo (p = {p_j[4]:.4f}).')
    print('Jumps contribuem para a previsao de volatilidade.')
else:
    print(f'\nCoeficiente de jumps (beta_j = {beta_j_val:.6f}) NAO e significativo (p = {p_j[4]:.4f}).')

# Visualizacao: Jump component
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(rv_data.index, jumps, color='red', alpha=0.6, linewidth=0.8)
axes[0].set_ylabel('Jump Component')
axes[0].set_title('Proxy de Jumps: max(RV_daily - RV_weekly, 0)')
axes[0].grid(True, alpha=0.3)

dates_reg = rv_data.index[max_lag + 1:max_lag + 1 + min_len]
axes[1].plot(dates_reg, y_har, alpha=0.5, linewidth=0.5, color='gray', label='RV Realizado')
axes[1].plot(dates_reg, fitted_har, alpha=0.7, linewidth=0.8, color='steelblue', label='HAR-RV')
axes[1].plot(dates_reg, fitted_harj, alpha=0.7, linewidth=0.8, color='red', label='HAR-RV-J')
axes[1].set_ylabel('Volatilidade Realizada')
axes[1].set_xlabel('Data')
axes[1].set_title('Ajuste: HAR-RV vs HAR-RV-J')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()